In [2]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [3]:
import numpy as np
import cv2
import tqdm
import pandas as pd
import glob
import os

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
import skimage.util as sk_noise

In [6]:
from pathlib import Path
import PIL as pil

In [19]:
np.random.seed(42)

In [7]:
!cp "/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/msu_iqa.zip" .
!unzip -qq msu_iqa.zip

In [9]:
!cp "/content/gdrive/MyDrive/DATABASES/CelebAspoof/face/ceas_iqa.zip" .
!unzip -qq ceas_iqa.zip

In [20]:
df_msu = pd.read_csv('msu_iqa.csv')
df_msu.head(1)

,frame,scene,client,label,data_type,full_path,iqa
0,frame_116.jpg,attack_client034_android_SD_printed_photo_scene01,client034,attack,train,fas_iqa/train/original/attack/attack_client034...,original


In [21]:
df_ceas = pd.read_csv('ceas_iqa.csv')
df_ceas.head(1)

,frame,scene,client,label,data_type,full_path,iqa
0,276917.jpg,client006191_Env0_Ilum0_Spt0,6191,real,train,fas_iqa/train/original/real/client006191_Env0_...,original


In [22]:
df_iqa = pd.concat([df_msu, df_ceas])
df_iqa = df_iqa.reset_index(drop=True)
print(df_iqa.shape[0])
df_iqa.sample(3)

4000


,frame,scene,client,label,data_type,full_path,iqa
555,frame_204.jpg,real_client002_laptop_SD_scene01,client002,real,train,fas_iqa/train/original/real/real_client002_lap...,original
3491,320844.jpg,client005600_Env1_Ilum2_Spt6,5600,attack,train,fas_iqa/train/original/attack/client005600_Env...,original
527,frame_125.jpg,real_client034_laptop_SD_scene01,client034,real,train,fas_iqa/train/original/real/real_client034_lap...,original


In [23]:
blur_types = {
   'BlurX': 0,
   'BlurY': 1,
   'BlurXY': 2,
   'GaussianBlur': 3,
}

blur_values = [7, 11, 15] #5
hbright_values = [5, 8, 10] #4
lbright_values = [.1, .2, .3] #5
noise_values = [15, 25, 45] #3
jpge_commpression_values = [10, 30, 50] #4

def add_moviment_blur(img_in, hor_vert_type=0, kernel_size=15):
   kernel = np.zeros((kernel_size, kernel_size))

   if hor_vert_type == 0:
       # Vertical kernel_v
       kernel[:, int((kernel_size - 1)/2)] = np.ones(kernel_size)
   else:
       # Horizontal kernel_h
       kernel[int((kernel_size - 1)/2), :] = np.ones(kernel_size)

   kernel /= kernel_size

   return cv2.filter2D(img_in, -1, kernel)


def add_blur_distortion(img_in=None, filter=1):
   # return cv2.blur(img_in, (filter, filter)) # Average blurring
   # return cv2.medianBlur(img_in, filter)   # Median blurring OK
   # return cv2.bilateralFilter(img_in, filter, 41, 21)   # Median blurring
   return cv2.GaussianBlur(img_in, (filter, filter), 0)  # Gaussian blurring, value to 0, automatically compute standar desviation based on our kernel size


def add_blur(img_in, blur_type=2, kernel=1):
   if blur_type<2:
       img_out = add_moviment_blur(img_in, hor_vert_type=blur_type, kernel_size=kernel)
   elif blur_type==2:
       img_out = add_moviment_blur(img_in, hor_vert_type=0, kernel_size=kernel)
       img_out = add_moviment_blur(img_out, hor_vert_type=1, kernel_size=kernel)
   else:
       img_out = add_blur_distortion(img_in, filter=kernel)

   return img_out


def add_bright_distortion(img_in=None, gamma=1):
   invGamma = 1.0 / gamma
   table = np.array([((i / 255.0) ** invGamma) * 255
                     for i in np.arange(0, 256)]).astype("uint8")
   img_out = cv2.LUT(img_in, table)
   return img_out


def add_noise_distortion(img_in=None, filter=1, divisor = 100):
   img_out = np.array(np.clip(sk_noise.random_noise(img_in, mode='gaussian', var=(filter / divisor) ** 2)*255, 0, 255),dtype=np.uint8)
   return img_out

In [24]:
!rm -r fas_iqa_dataset/BlurX/
!rm -r fas_iqa_dataset/BlurY/
!rm -r fas_iqa_dataset/BlurXY/
!rm -r fas_iqa_dataset/GaussianBlur/
!rm -r fas_iqa_dataset/hbright/
!rm -r fas_iqa_dataset/lbright/
!rm -r fas_iqa_dataset/noise/
!rm -r fas_iqa_dataset/jpgcompression/
!rm -r fas_iqa_dataset/original/

output_file = 'fas_iqa_dataset/'
output_zip = f"{output_file.replace('/', '')}.zip"
for idx, row in tqdm.tqdm(df_iqa.iterrows(), total=len(df_iqa)):
    path_img = row['full_path']

    filename = row['frame']

    scene = row['scene']
    client = row['client']
    label = row['label']
    data_type = row['data_type']


    img = cv2.imread(path_img)
    img = cv2.resize(img, (256, 256))

    part_path_end = f"{data_type}/{label}/{scene}"

    Path(f'{output_file}/original/{part_path_end}/').mkdir(parents=True, exist_ok=True)
    cv2.imwrite(f'{output_file}/original/{part_path_end}/{filename}', img)

    for blur_type in blur_types:
        for blur_value in blur_values:
            img_blur = add_blur(img, blur_type=blur_types[blur_type], kernel=blur_value)
            Path(f'{output_file}/{blur_type}/{part_path_end}/').mkdir(parents=True, exist_ok=True)
            cv2.imwrite(f'{output_file}/{blur_type}/{part_path_end}/{blur_type}_{blur_value}_f_{filename}', img_blur)

    Path(f'{output_file}/hbright/{part_path_end}/').mkdir(parents=True, exist_ok=True)
    for hbright_value in hbright_values:
        img_hbright = add_bright_distortion(img,gamma=hbright_value)
        cv2.imwrite(f'{output_file}/hbright/{part_path_end}/hbright_{hbright_value}_f_{filename}', img_hbright)

    Path(f'{output_file}/lbright/{part_path_end}/').mkdir(parents=True, exist_ok=True)
    for lbright_value in lbright_values:
        img_lbright = add_bright_distortion(img,gamma=lbright_value)
        cv2.imwrite(f'{output_file}/lbright/{part_path_end}/lbright_{lbright_value}_f_{filename}', img_lbright)

    Path(f'{output_file}/noise/{part_path_end}/').mkdir(parents=True, exist_ok=True)
    for noise_value in noise_values:
        img_noise = add_noise_distortion(img, filter=noise_value)
        cv2.imwrite(f'{output_file}/noise/{part_path_end}/noise_{noise_value}_f_{filename}', img_noise)

    Path(f'{output_file}/jpgcompression/{part_path_end}/').mkdir(parents=True, exist_ok=True)
    for jpge_commpression_value in jpge_commpression_values:
        pil.Image.fromarray(img[:,:,::-1]).save(f'{output_file}/jpgcompression/{part_path_end}/jpgcompression_{jpge_commpression_value}_f_{filename}', optimize=True, quality=jpge_commpression_value)


    # break

100%|██████████| 4000/4000 [06:41<00:00,  9.97it/s]


In [25]:
!zip -r {output_zip} {output_file} -q

In [26]:
!cp {output_zip} "/content/gdrive/MyDrive/DATABASES/iqa_spoof_dataset/"

END